In [7]:
import json
from pathlib import Path

from rdflib import Graph
from rdfine import GraphReader
from compilers import PipelineGenerator, ProjectBuilder

In [8]:
graph = Graph()

data_dir = Path("../data")
files = [
    "catalog.ttl",
    "pipeline_definition_nifi.ttl",
    "tcs_shapes.ttl",
]

secrets_file = data_dir / "pipeline_definition_nifi.secrets.ttl"
if secrets_file.exists():
    files.append(secrets_file.name)
    
for filename in files:
    graph.parse(
        data_dir / filename,
        publicID="file:///workspace/pipeline/",
    )

reader = GraphReader(graph).infer(data_dir / "inference_rules.yaml")

In [9]:
report = reader.validate(advanced=True, inference="rdfs")

violations = report.select(
    "?focus ?message",
    """
    ?result a sh:ValidationResult ;
        sh:focusNode ?focus ;
        sh:resultMessage ?message .
    """,
)

for row in violations.itertuples(index=False):
    print(f"Focus:   {row.focus}")
    print(f"Message: {row.message}")
    print()
    
if not report.ask("?report sh:conforms true"):
    for row in violations.itertuples(index=False):
        print(f"{row.focus}: {row.message}")

    raise ValueError("Pipeline definition does not conform")

In [10]:
generator = PipelineGenerator(":DemonstratorPipeline", reader.graph)
build_graph = generator.compile()

[compiler.__name__ for compiler in generator.compilers]

['PipelineExtractor',
 'PipelineAssembler',
 'NifiConfigCompiler',
 'DockerComposeCompiler']

In [11]:
builder = ProjectBuilder(build_graph)

for _, file in builder.files.iterrows():
    content = file["content"]
    if file["filename"] == "flow.json":
        flow = json.loads(content)
        for kind in ("processors", "controllerServices"):
            for component in flow["rootGroup"][kind]:
                for name, descriptor in component["propertyDescriptors"].items():
                    if descriptor["sensitive"] and name in component["properties"]:
                        component["properties"][name] = "[REDACTED]"
        content = json.dumps(flow, indent=4)

    print(f"=== {file['filepath']}/{file['filename']} ===")
    print(content)

=== ./docker-compose.yml ===
services:
  nifip:
    container_name: nifip
    image: apache/nifi:2.10.0
    environment:
      SINGLE_USER_CREDENTIALS_USERNAME: root
      SINGLE_USER_CREDENTIALS_PASSWORD: rootrootrootroot
      NIFI_SENSITIVE_PROPS_KEY: dishacled-nifi-dev-key
    ports:
    - 8443:8443
    volumes:
    - ./nifi/flow.json:/opt/nifi/bootstrap/flow.json:ro
    entrypoint:
    - /bin/bash
    - -c
    - |-
      set -e
      gzip -c /opt/nifi/bootstrap/flow.json > /opt/nifi/nifi-current/conf/flow.json.gz
      exec /opt/nifi/scripts/start.sh

=== nifi/flow.json ===
{
    "maxTimerDrivenThreadCount": 10,
    "rootGroup": {
        "identifier": "92b7e0fa-f01d-50ba-a3f9-d4b952e8a26b",
        "instanceIdentifier": "c36b4aa0-6ae8-5922-99d3-7cf17834cd86",
        "name": "Demonstrator Pipeline.",
        "comments": "Polls API, transforms to RDF, detects threshold, triggers email alert.",
        "position": {
            "x": 0.0,
            "y": 0.0
        },
        "pro

In [12]:
written = builder.write("../out/nifi_testing")

for path in written:
    print(path)

C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\nifi_testing\docker-compose.yml
C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\nifi_testing\nifi\flow.json
